# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

feature_vector = duckdb.sql("""
    WITH prior_position AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.word_count,
        d.search_volume,
        d.competition_level,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-01') AS content_age_days,
        p.gsc_avg_position_prior
    FROM dim_content d
    LEFT JOIN prior_position p
        USING (client_hash_id, content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
      AND d.content_created_date <= DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Missing value handling ---
# word_count: missing is tied to provider_used being NULL (an older ingestion gap),
# not "this page has no words" — so fill with median AND keep a flag, never fill with 0.
feature_vector["word_count_missing"] = feature_vector["word_count"].isna().astype(int)
feature_vector["word_count"] = feature_vector["word_count"].fillna(feature_vector["word_count"].median())

In [ ]:
# gsc_avg_position_prior: missing means "no GSC history in February" — a real, structural
# gap (young page or no GSC access), not "position is average." Flag it, then fill.
feature_vector["gsc_position_missing"] = feature_vector["gsc_avg_position_prior"].isna().astype(int)
feature_vector["gsc_avg_position_prior"] = feature_vector["gsc_avg_position_prior"].fillna(
    feature_vector["gsc_avg_position_prior"].median()
)

In [ ]:
# --- Categorical handling ---
# competition_level is ordinal (LOW < MEDIUM < HIGH), not unordered — map it to numbers
# rather than one-hot encoding, so the model can use the natural order.
competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
feature_vector["competition_level_encoded"] = feature_vector["competition_level"].map(competition_map)

feature_vector.head()

,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior,word_count_missing,gsc_position_missing,competition_level_encoded
0,client_0797ff3a1fc9a6a5,content_9323fd059ff65ad0,3869,30,LOW,145,15.375000,0,0,0.0
1,client_0797ff3a1fc9a6a5,content_9bb9c6a63fc2003f,3950,20,LOW,145,9.195513,0,0,0.0
2,client_0797ff3a1fc9a6a5,content_9d5a482ec16ea617,4067,10,LOW,145,9.181159,0,0,0.0
3,client_0797ff3a1fc9a6a5,content_9ed024c40862c4aa,3763,30,LOW,145,3.000000,0,0,0.0
4,client_0797ff3a1fc9a6a5,content_a0a6b37ae2f9a09c,3075,0,LOW,145,14.690257,0,0,0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Rebuild label: did clicks decline in March vs. February?
labels = duckdb.sql("""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_march
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT feb.client_hash_id, feb.content_hash_id, clicks_feb, clicks_march,
           CASE WHEN clicks_march < clicks_feb THEN 1 ELSE 0 END AS declined
    FROM feb JOIN march USING (client_hash_id, content_hash_id)
""").df()

data = feature_vector.merge(labels, on=["client_hash_id", "content_hash_id"])

honest_cols = ["word_count", "word_count_missing", "search_volume",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
print(data[honest_cols].isna().sum())

word_count                       0
word_count_missing               0
search_volume                10684
competition_level_encoded    11279
content_age_days                 0
gsc_avg_position_prior           0
gsc_position_missing             0
dtype: int64


In [ ]:
# search_volume: missing means no keyword-volume data was ever recorded for this page.
# Flag it, then fill with median (0 would wrongly imply "definitely no search demand").
data["search_volume_missing"] = data["search_volume"].isna().astype(int)
data["search_volume"] = data["search_volume"].fillna(data["search_volume"].median())

# competition_level_encoded: NaN here means competition_level was NULL or an unmapped
# value (not LOW/MEDIUM/HIGH) — this is categorical, so use -1 as "unknown", not median.
data["competition_level_encoded"] = data["competition_level_encoded"].fillna(-1)

# Confirm all gaps are gone
print(data[honest_cols + ["search_volume_missing"]].isna().sum())

word_count                   0
word_count_missing           0
search_volume                0
competition_level_encoded    0
content_age_days             0
gsc_avg_position_prior       0
gsc_position_missing         0
search_volume_missing        0
dtype: int64


In [ ]:
honest_cols = ["word_count", "word_count_missing", "search_volume", "search_volume_missing",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]

In [ ]:
X = data[honest_cols]
y = data["declined"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC:", honest_auc)


Honest AUC: 0.6218456225480447


In [ ]:
# Attack 1: raw same-window value
data["clicks_march_test"] = data["clicks_march"]
X_attack1 = data[honest_cols + ["clicks_march_test"]]
X_tr, X_te, y_tr, y_te = train_test_split(X_attack1, y, test_size=0.3, random_state=42)
m1 = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print("Attack 1 (clicks_march) AUC:", roc_auc_score(y_te, m1.predict_proba(X_te)[:, 1]))

# Attack 2: the actual label-derived column
data["clicks_diff_test"] = data["clicks_march"] - data["clicks_feb"]
X_attack2 = data[honest_cols + ["clicks_diff_test"]]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_attack2, y, test_size=0.3, random_state=42)
m2 = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
print("Attack 2 (clicks_diff) AUC:", roc_auc_score(y_te2, m2.predict_proba(X_te2)[:, 1]))

Attack 1 (clicks_march) AUC: 0.6219477607408156
Attack 2 (clicks_diff) AUC: 1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = {
    "client_hash_id / content_hash_id / keyword_hash_id / url_hash_id": "identifiers — used for joining/grouping only, never a learnable pattern",
    "provider_used / model_used": "describes internal content-authoring tooling, not a real signal about page performance",
    "is_deleted": "lifecycle/product-decision flag, used only to filter the pool, not a feature",
    "clicks_march / gsc_clicks (March window)": "inside the label window — this is what the label is computed from, using it as a feature is direct leakage",
    "clicks_diff": "mathematically equivalent to the label itself, demonstrated in the leakage hunt above",
    "ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta / ai_other": "overly granular per-AI-source breakdown; risks overlapping with future windows, aggregate sessions_ai is sufficient for this lane",
    "client_has_gsc / client_has_ga4 / gsc_data_available / ga4_data_available": "used to filter which rows are eligible for scoring, not fed to the model as a predictive signal",
}
for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

- client_hash_id / content_hash_id / keyword_hash_id / url_hash_id: identifiers — used for joining/grouping only, never a learnable pattern
- provider_used / model_used: describes internal content-authoring tooling, not a real signal about page performance
- is_deleted: lifecycle/product-decision flag, used only to filter the pool, not a feature
- clicks_march / gsc_clicks (March window): inside the label window — this is what the label is computed from, using it as a feature is direct leakage
- clicks_diff: mathematically equivalent to the label itself, demonstrated in the leakage hunt above
- ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta / ai_other: overly granular per-AI-source breakdown; risks overlapping with future windows, aggregate sessions_ai is sufficient for this lane
- client_has_gsc / client_has_ga4 / gsc_data_available / ga4_data_available: used to filter which rows are eligible for scoring, not fed to the model as a predictive signal


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.